### 项目总结：基于轻量级过程奖励模型 (PRM) 的数学推理评估

本项目旨在通过构建和评估轻量级的过程奖励模型（PRM），来优化数学推理轨迹的打分机制。我们利用冻结的语义编码器进行特征预计算，在此基础上快速训练并对比了多种轻量级神经网络架构（MLP、CNN、GRU）在成对 Q 值匹配（PQM）损失函数下的表现。

#### 核心项目流程
*   **阶段一：特征预计算 (Stage 1)** —— 利用冻结的 Qwen2.5-0.5B 模型，对全量数学数据进行前向传播，提取并缓存隐藏层特征。
*   **阶段二：轻量级网络头构建 (Stage 2)** —— 设计并实例化了小参数量、高效率的对比网络架构，用于接收缓存的特征。
*   **阶段三：对比训练与评估 (Stage 3)** —— 在轨迹级别进行优化训练，并对模型的打分排序能力进行多维度测试。

---

#### 1. 数据准备与防泄漏策略
为了确保模型评估的绝对严谨，我们将庞大的 Math-Shepherd 数据集严格按 99% 与 1% 的比例，切分为了互不重合的训练集与测试集，从根源上杜绝了数据泄露。

在处理这四十多万条数据的切分时，我们遭遇并克服了 Windows 系统经典的 I/O 缓冲问题。由于数据量极其庞大，常规的简短指令会导致数据滞留在内存中无法完全写入硬盘。我们通过引入强制落盘和清理缓冲区的机制，确保了大规模数据集的物理级安全写入。

#### 2. 奖励网络训练与调优
我们基于纯净的特征缓存，让三种不同的网络架构在完全相同的起跑线上进行了训练。

在初期，模型遭遇了深度学习中常见的梯度爆炸问题（Loss 变为 NaN）。经过排查，我们将初始学习率下调了十倍。这一关键超参数的修复瞬间稳住了训练阵脚，三种模型随后均在 3 个 Epoch 内呈现出健康的 Loss 下降趋势并成功收敛。

#### 3. 单步轨迹评估 (期中测验)
在完成训练后，我们对模型进行了成对分离度（Pairwise Separation）的测试，即评估模型将“正确步骤”的打分排在“错误步骤”前面的概率。在这个过程中，我们排除了两个工程隐患：
*   **兼容性修复**：统一了不同数据集格式下的标签键名差异。
*   **维度冲突修复**：由于数学题的解答步数长短不一，张量拼接时会发生崩溃。我们通过引入掩码逻辑，精准提取了每道题“最后一个有效步骤”的标签，成功将步骤级数据转化为整题级数据，打通了评估链路。

**评估结果对比**

| 网络架构 | 参数量级 | 成对分离度 (Pairwise Separation) | 架构特点与结论 |
| :--- | :--- | :--- | :--- |
| **MLP** | 约 26 万 | **0.6905** | 作为基线前馈网络，表现达标。 |
| **CNN** | 约 39 万 | **0.7067** | 能够捕捉局部相邻步骤的关联，分数获得显著提升。 |
| **GRU** | 约 79 万 | **0.7112** | **全场最佳。** 证明了具有时序记忆功能的循环网络，天生最适合处理数学推导这种高度依赖前后文的串行逻辑任务。 |

> **学术洞察：**
> 虽然 GRU 相比 MLP 在绝对准确率上仅提升了约 2%，但这代表它在随机基线（50%）之上，多挖掘出了超过 10% 的有效特征。在动辄十几步的复杂数学推理中，这种对序列依赖的精准把控，能极大地降低误差的指数级累积。

#### 4. Best-of-N (BoN) 终极评估 (待推进)
为了对齐 PRM 领域的最高学术标准，我们正着手推进 Best-of-N 评估，即测试模型能否从同一道题的 N 种不同解答路径中，精准挑出唯一正确的答案。

目前我们已经修复了 Windows 系统默认编码导致的文本读取乱码问题。当前面临的主要阻点在于数据结构不匹配：BoN 评估要求数据集必须是“按题分组”的层级化结构（包含候选解答列表），而我们目前使用的是随机切分的扁平化数据。接下来的计划是寻找匹配的专属 BoN 测试集，或者通过重构脚本，将现有测试集强行按题目进行聚类转换，以完成最终的测试。

In [1]:
from datasets import load_dataset
import random
import json

# 抽取数量（100~200之间任选）
N = 150

# 固定随机种子，保证结果可复现
SEED = 42

# 加载训练集
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 随机抽样
random.seed(SEED)
indices = random.sample(range(len(dataset)), N)

# 保存为 JSONL
with open("dummy_train.jsonl", "w", encoding="utf-8") as f:
    for idx in indices:
        json.dump(dataset[idx], f, ensure_ascii=False)
        f.write("\n")

print(f"Saved {N} samples to dummy_train.jsonl")

f:\anaconda3\envs\node2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 150 samples to dummy_train.jsonl


In [2]:
from datasets import load_dataset
import json

# 加载 Math-Shepherd 数据集（请替换为具体的 HF 仓库名，如 "math-shepherd/..."）
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 导出为你代码需要的 jsonl 格式
output_file = "data/math_shepherd_full.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
print(f"下载完成！共 {len(dataset)} 条数据，已保存至 {output_file}")

KeyboardInterrupt: 